In [ ]:
import requests, time

KAKAO_REST_KEY = "본인 REST API 키"
url = "https://dapi.kakao.com/v2/local/search/address.json"

def jibun_to_road(jibun):
    headers = {"Authorization": f"KakaoAK : YOUR_API_KEY_HERE}"}
    params  = {"query": jibun}
    r = requests.get(url, headers=headers, params=params, timeout=5)
    r.raise_for_status()
    docs = r.json()["documents"]
    if not docs:
        return None
    # ① 좌표 얻기
    x, y = docs[0]["x"], docs[0]["y"]

    # ② 좌표→도로명주소
    url2 = "https://dapi.kakao.com/v2/local/geo/coord2address.json"
    r2 = requests.get(url2, headers=headers, params={"x":x, "y":y}, timeout=5)
    r2.raise_for_status()
    road = r2.json()["documents"][0]["road_address"]
    return road["address_name"] if road else None

# ------------------------------
# 대량 변환 – 0.1초 딜레이로 1만 건 ≈ 17분
converted = []
for j in df["jibun_addr"]:
    converted.append(jibun_to_road(j))
    time.sleep(0.1)          # QPS 제한 대비
df["road_addr"] = converted
df.to_csv("산불데이터_도로명주소.csv", index=False, encoding="utf-8-sig")


SyntaxError: invalid decimal literal (<ipython-input-1-814739eea0c0>, line 7)

In [ ]:
import requests, time, pandas as pd

API_KEY = "YOUR_API_KEY_HERE"
KAKAO_URL = "https://dapi.kakao.com/v2/local/search/address.json"
HEADERS   = {"Authorization": f"KakaoAK {API_KEY}"}

def kakao_geocode(q, timeout=5):
    r = requests.get(KAKAO_URL, headers=HEADERS, params={'query': q}, timeout=timeout)
    r.raise_for_status()
    docs = r.json()['documents']
    if docs:            # 가장 첫 결과만 사용
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def make_variants(si, gun, eupmyeon, dong):
    # 1단계
    base = f"{si} {gun} {eupmyeon} {dong}"
    yield base
    # 2단계: 군/시 보정
    gun2 = gun if gun.endswith(('시','군','구')) else gun + ('시' if '시' in si else '군')
    yield f"{si} {gun2} {eupmyeon} {dong}"
    # 3단계: 읍·면·동 후보
    if not eupmyeon.endswith(('읍','면','동')):
        for suf in ['읍','면','동']:
            yield f"{si} {gun2} {eupmyeon}{suf} {dong}"
    # 4단계: 리 보정
    if not dong.endswith('리'):
        yield f"{si} {gun2} {eupmyeon} {dong}리"

def safe_geocode(si, gun, eupmyeon, dong):
    for cand in make_variants(si, gun, eupmyeon, dong):
        lat, lon = kakao_geocode(cand)
        if lat:          # 성공
            return lat, lon, cand
    return None, None, None

# ---------- 파일 처리 ----------
df = pd.read_csv("./산불데이터20160101_20250425.csv", encoding="utf-8")

lat_list, lon_list, used_addr = [], [], []
for i, row in df.iterrows():
    lat, lon, cand = safe_geocode(row['locsi'], row['locgungu'],
                                  row['locmenu'], row['locdong'])
    lat_list.append(lat)
    lon_list.append(lon)
    used_addr.append(cand)
    if i % 50 == 0:             # 진행 상황 로그
        print(f"{i} done")
    time.sleep(0.07)            # 14 req/s ≈ 840 req/min < 100 req/min 안전선

df['위도'] = lat_list
df['경도'] = lon_list
df['검색에_성공한_주소'] = used_addr
df.to_csv("/mnt/data/산불데이터_위경도추가.csv", index=False, encoding="utf-8-sig")


0 done


AttributeError: 'float' object has no attribute 'endswith'

In [ ]:
import requests

#카카오 API 키
api_key = "YOUR_API_KEY_HERE"

def address_to_coords(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {
        "Authorization": f"KakaoAK {api_key}"
    }
    params = {
        "query": address
    }

    response = requests.get(url, headers=headers, params=params)
    result = response.json()

    if result['documents']:
        x = result['documents'][0]['x']  # 경도
        y = result['documents'][0]['y']  # 위도
        return x, y
    else:
        return None, None

# 예시
address = "서울특별시 강남구 테헤란로 123"
longitude, latitude = address_to_coords(address)
print(f"위도: {latitude}, 경도: {longitude}")


위도: 37.4995539438207, 경도: 127.031393491745


In [ ]:
import requests

#카카오 API 키
api_key = "YOUR_API_KEY_HERE"

def address_to_coords(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {
        "Authorization": f"KakaoAK {api_key}"
    }
    params = {
        "query": address
    }

    response = requests.get(url, headers=headers, params=params)
    result = response.json()

    if result['documents']:
        x = result['documents'][0]['x']  # 경도
        y = result['documents'][0]['y']  # 위도
        return x, y
    else:
        return None, None

# 예시
address = "충북 청주시 가덕면"
longitude, latitude = address_to_coords(address)
print(f"위도: {latitude}, 경도: {longitude}")


위도: 36.5535696977221, 경도: 127.548827630272


In [ ]:
import requests

#카카오 API 키
api_key = "YOUR_API_KEY_HERE"

def address_to_coords(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {
        "Authorization": f"KakaoAK {api_key}"
    }
    params = {
        "query": address
    }

    response = requests.get(url, headers=headers, params=params)
    result = response.json()

    if result['documents']:
        x = result['documents'][0]['x']  # 경도
        y = result['documents'][0]['y']  # 위도
        return x, y
    else:
        return None, None

# 예시
address = "경상북도 군위"
longitude, latitude = address_to_coords(address)
print(f"위도: {latitude}, 경도: {longitude}")


위도: None, 경도: None


In [ ]:
import pandas as pd
import requests, time

API_KEY = "YOUR_API_KEY_HERE"
SRC      = "./산불데이터_with_addr_simple.csv"   # 입력
OUT      = "./산불데이터_geo_sample10.csv"      # 출력 (10개 샘플)

# 1) 데이터 읽기
df = pd.read_csv(SRC, encoding="utf-8")

# 2) 상위 10개만 추출
sample = df.head(10).copy()

# 3) 위·경도 함수
def address_to_coords(addr: str):
    url     = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {API_KEY}"}
    try:
        resp = requests.get(url, headers=headers, params={"query": addr}, timeout=5)
        resp.raise_for_status()
        docs = resp.json().get("documents", [])
        if docs:
            return float(docs[0]["y"]), float(docs[0]["x"])   # (lat, lon)
    except Exception:
        pass
    return None, None     # 실패 시

# 4) 주소 → 좌표
lats, lons = [], []
for addr in sample["간략주소"]:
    lat, lon = address_to_coords(addr)
    lats.append(lat)
    lons.append(lon)
    time.sleep(0.2)       # 초당 5회 이하 호출 – 안전

sample["lat"] = lats
sample["lon"] = lons

# 5) 결과 저장 & 확인
sample.to_csv(OUT, index=False, encoding="utf-8-sig")
print(sample[["간략주소", "lat", "lon"]])
print(f"\n✅ 저장 완료 → {OUT}")


        간략주소        lat         lon
0   강원 홍천 내촌  37.816348  128.086740
1  강원 강릉 주문진  37.894000  128.823429
2  강원 강릉 주문진  37.894000  128.823429
3   전남 보성 율어  34.870589  127.186972
4   경북 고령 개진  35.708114  128.350137
5   강원 횡성 안흥  37.412513  128.156050
6   전남 함평 대동  35.068980  126.531399
7  강원 태백 하사미  37.310626  128.985358
8  경기 남양주 화도  37.657910  127.300352
9   전북 부안 진서  35.587643  126.605433

✅ 저장 완료 → ./산불데이터_geo_sample10.csv


In [ ]:
import pandas as pd, requests, time, os, json
from pathlib import Path

# === 설정 ===
API_KEY = "YOUR_API_KEY_HERE"
SRC      = "./산불데이터_with_addr_simple.csv"   # 입력 파일
OUT      = "./산불데이터_with_geo.csv"           # 최종 출력
CHKPT    = "./_geo_tmp.json"                    # 중간 체크포인트
SLEEP    = 0.12   # 초당 8~9 건 정도 (30 건/초 제한 대비 여유)

# === 1) 데이터 읽기 ===
df = pd.read_csv(SRC, encoding="utf-8")

# === 2) 체크포인트 불러오기(있으면) ===
done = {}
if os.path.exists(CHKPT):
    with open(CHKPT, "r", encoding="utf-8") as f:
        done = json.load(f)          # {"주소":"lat,lon or null"}

def address_to_coords(addr):
    """Kakao 주소검색 → (lat, lon) or (None, None)"""
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {API_KEY}"}
    try:
        r = requests.get(url, headers=headers, params={"query": addr}, timeout=5)
        r.raise_for_status()
        docs = r.json().get("documents", [])
        if docs:
            return float(docs[0]["y"]), float(docs[0]["x"])
    except Exception:
        pass
    return None, None

# === 3) 변환 루프 ===
lats, lons = [], []
for i, addr in enumerate(df["간략주소"]):
    # (1) 캐시/체크포인트 먼저 확인
    if addr in done:
        lat, lon = done[addr] or (None, None)
    else:
        lat, lon = address_to_coords(addr)
        done[addr] = (lat, lon) if lat is not None else None
        # 주기적 체크포인트(100 건마다 저장)
        if i % 100 == 0:
            with open(CHKPT, "w", encoding="utf-8") as f:
                json.dump(done, f, ensure_ascii=False)
        time.sleep(SLEEP)
    lats.append(lat)
    lons.append(lon)

# === 4) 결과 병합 & 저장 ===
df["lat"] = lats
df["lon"] = lons
df.to_csv(OUT, index=False, encoding="utf-8-sig")

# === 5) 체크포인트 정리 ===
if os.path.exists(CHKPT):
    os.remove(CHKPT)

print(f"✅ 완료! → {OUT}")
print(df[["간략주소", "lat", "lon"]].head())


✅ 완료! → ./산불데이터_with_geo.csv
        간략주소        lat         lon
0   강원 홍천 내촌  37.816348  128.086740
1  강원 강릉 주문진  37.894000  128.823429
2  강원 강릉 주문진  37.894000  128.823429
3   전남 보성 율어  34.870589  127.186972
4   경북 고령 개진  35.708114  128.350137


In [ ]:
import pandas as pd, requests, time, os, json
from pathlib import Path

# === 설정 ===
API_KEY  = "YOUR_API_KEY_HERE"
SRC      = "./산불데이터_geo_missing_simple2_fix.csv"  # 입력
OUT      = "./산불데이터_geo_missing_fix_coord.csv"   # 출력
CHKPT    = "./_coord_tmp.json"                        # 중간 캐시
SLEEP    = 0.12   # 초당 8~9건 → 30 req/sec 한도 대비

# === 1. 데이터 읽기 ===
df = pd.read_csv(SRC, encoding="utf-8")

# === 2. 체크포인트 불러오기 ===
cache = {}
if os.path.exists(CHKPT):
    with open(CHKPT, "r", encoding="utf-8") as f:
        cache = json.load(f)        # {"주소":"lat,lon" or null}

# === 3. Kakao 주소검색 함수 ===
def kakao_coords(addr):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {API_KEY}"}
    try:
        r = requests.get(url, headers=headers, params={"query": addr}, timeout=5)
        r.raise_for_status()
        docs = r.json().get("documents", [])
        if docs:
            return float(docs[0]["y"]), float(docs[0]["x"])
    except Exception:
        pass
    return None, None

# === 4. 변환 루프 ===
lats, lons = [], []
for i, addr in enumerate(df["초간략주소_보정"]):
    if addr in cache:                          # 캐시 HIT
        lat, lon = cache[addr] or (None, None)
    else:                                      # API 호출
        lat, lon = kakao_coords(addr)
        cache[addr] = (lat, lon) if lat else None
        if i % 100 == 0:                       # 100건마다 캐시 파일 저장
            with open(CHKPT, "w", encoding="utf-8") as f:
                json.dump(cache, f, ensure_ascii=False)
        time.sleep(SLEEP)
    lats.append(lat)
    lons.append(lon)

# === 5. 결과 병합 & 저장 ===
df["lat2"] = lats
df["lon2"] = lons
df.to_csv(OUT, index=False, encoding="utf-8-sig")

# === 6. 캐시 파일 정리 ===
if os.path.exists(CHKPT):
    os.remove(CHKPT)

print("✅ 완료 →", OUT)
print(df[["초간략주소_보정", "lat2", "lon2"]].head())


✅ 완료 → ./산불데이터_geo_missing_fix_coord.csv
  초간략주소_보정       lat2        lon2
0    강원 홍천  37.697167  127.888684
1    전남 여수  34.760401  127.662331
2    강원 삼척  37.449898  129.165112
3    광주 남구  35.132819  126.902476
4    대구 북구  35.885683  128.582781
